In [322]:
import numpy as np
import pandas as pd
import re,ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [323]:
df = pd.read_csv('appartments.csv')

In [324]:
df.iloc[22]

PropertyName                PropertyName
PropertySubName          PropertySubName
NearbyLocations          NearbyLocations
LocationAdvantages    LocationAdvantages
Link                                Link
PriceDetails                PriceDetails
TopFacilities              TopFacilities
Name: 22, dtype: object

In [325]:
df=df.drop(22)

In [326]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."


In [327]:
df.shape

(246, 7)

In [328]:
df.iloc[2].NearbyLocations

"['AIPL Business Club Sector 62', 'Heritage Xperiential Learning School', 'CK Birla Hospital', 'Paras Trinity Mall Sector 63', 'Rapid Metro Station Sector 56']"

In [329]:
df.iloc[2].LocationAdvantages

"{'AIPL Business Club Sector 62': '2.7 Km', 'Heritage Xperiential Learning School': '2 Km', 'CK Birla Hospital': '2.5 Km', 'Paras Trinity Mall Sector 63': '3.5 Km', 'Rapid Metro Station Sector 56': '3.8 Km', 'De Adventure Park': '6.8 Km', 'Golf Course Ext Rd': '99 Meter', 'DoubleTree by Hilton Hotel Gurgaon': '3.6 Km', 'KIIT College of Engineering Sohna Road': '8.4 Km', 'Mehrauli-Gurgaon Road': '11.8 Km', 'Indira Gandhi International Airport': '21.1 Km', 'Nirvana Rd': '160 Meter', 'TERI Golf Course': '8.7 Km'}"

In [330]:
df.iloc[1].PriceDetails

"{'3 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '1,605 - 2,170 sq.ft.', 'price-range': '₹ 2.2 - 3.03 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,248 - 2,670 sq.ft.', 'price-range': '₹ 3.08 - 3.73 Cr'}}"

In [331]:
df.iloc[2].TopFacilities

"['Terrace Garden', 'Gazebo', 'Fountain', 'Amphitheatre', 'Party Lawn', 'Basketball Court', 'Badminton Court', 'Yoga/Meditation Area', 'Indoor Games']"

In [332]:
df[['PropertyName','TopFacilities']]

,PropertyName,TopFacilities
0,Smartworld One DXP,"['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."
...,...,...
242,DLF Princeton Estate,"['Swimming Pool', 'Medical Centre', 'Laundry',..."
243,Pyramid Urban Homes 2,"['Shopping Centre', 'Community Hall', '24x7 Se..."
244,Satya The Hermitage,"['Bus Shelter', 'Swimming Pool', 'Business Lou..."
245,BPTP Spacio,"['Swimming Pool', 'Card Room', 'Piped Gas', 'P..."


In [333]:
df[['PropertyName','TopFacilities']]['TopFacilities'][0]

"['Swimming Pool', 'Salon', 'Restaurant', 'Spa', 'Cafeteria', 'Sun Deck', '24x7 Security', 'Club House', 'Gated Community']"

In [334]:
def extract_list(s):
    return re.findall(r"'(.*?)'", s)

df['TopFacilities'] = df['TopFacilities'].apply(extract_list)

In [335]:
df[['PropertyName','TopFacilities']]['TopFacilities'][0]

['Swimming Pool',
 'Salon',
 'Restaurant',
 'Spa',
 'Cafeteria',
 'Sun Deck',
 '24x7 Security',
 'Club House',
 'Gated Community']

In [336]:
df['FacilitiesStr'] = df['TopFacilities'].apply(' '.join)

In [337]:
df['FacilitiesStr'][0]

'Swimming Pool Salon Restaurant Spa Cafeteria Sun Deck 24x7 Security Club House Gated Community'

In [338]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [339]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df['FacilitiesStr'])

In [340]:
tfidf_matrix.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.14379378,
        0.14379378],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(246, 953))

In [341]:
tfidf_matrix.toarray().shape

(246, 953)

In [342]:
tfidf_matrix.toarray()[0]

array([0.        , 0.        , 0.        , 0.18809342, 0.18809342,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [343]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [344]:
cosine_sim1.shape

(246, 246)

In [345]:

def recommend_properties(property_name, cosine_sim=cosine_sim1):
    # Get the index of the property that matches the name
    idx = df.index[df['PropertyName'] == property_name].tolist()[0]

    # Get the pairwise similarity scores with that property
    sim_scores = list(enumerate(cosine_sim1[idx]))

    # Sort the properties based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar properties
    sim_scores = sim_scores[1:6]

    # Get the property indices
    property_indices = [i[0] for i in sim_scores]
    
    recommendations_df = pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[property_indices],
        'SimilarityScore': sim_scores
    })

    # Return the top 10 most similar properties
    return recommendations_df

In [346]:
recommend_properties("DLF The Arbour")

,PropertyName,SimilarityScore
64,Ace Palm Floors,"(63, 0.4529382062441955)"
217,Yashika 104,"(216, 0.4199606322926784)"
93,JMS The Nation,"(92, 0.4166584649363288)"
154,India Rashtra,"(153, 0.398954234680194)"
0,Smartworld One DXP,"(0, 0.38885046199432893)"


In [347]:
df[['PropertyName','PriceDetails']]

,PropertyName,PriceDetails
0,Smartworld One DXP,"{'2 BHK': {'building_type': 'Apartment', 'area..."
1,M3M Crown,"{'3 BHK': {'building_type': 'Apartment', 'area..."
2,Adani Brahma Samsara Vilasa,{'3 BHK': {'building_type': 'Independent Floor...
3,Sobha City,"{'2 BHK': {'building_type': 'Apartment', 'area..."
4,Signature Global City 93,{'2 BHK': {'building_type': 'Independent Floor...
...,...,...
242,DLF Princeton Estate,"{'2 BHK': {'building_type': 'Apartment', 'area..."
243,Pyramid Urban Homes 2,"{'1 BHK': {'building_type': 'Apartment', 'area..."
244,Satya The Hermitage,"{'2 BHK': {'building_type': 'Apartment', 'area..."
245,BPTP Spacio,"{'2 BHK': {'building_type': 'Apartment', 'area..."


In [348]:
df[['PropertyName','PriceDetails']]['PriceDetails'][1]

"{'3 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '1,605 - 2,170 sq.ft.', 'price-range': '₹ 2.2 - 3.03 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,248 - 2,670 sq.ft.', 'price-range': '₹ 3.08 - 3.73 Cr'}}"

In [349]:
import pandas as pd
import json

# Load the dataset
df_appartments = pd.read_csv('appartments.csv').drop(22)

# Function to parse and extract the required features from the PriceDetails column
def refined_parse_modified_v2(detail_str):
    try:
        details = json.loads(detail_str.replace("'", "\""))
    except:
        return {}

    extracted = {}
    for bhk, detail in details.items():
        # Extract building type
        extracted[f'building type_{bhk}'] = detail.get('building_type')

        # Parsing area details
        area = detail.get('area', '')
        area_parts = area.split('-')
        if len(area_parts) == 1:
            try:
                value = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area low {bhk}'] = value
                extracted[f'area high {bhk}'] = value
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None
        elif len(area_parts) == 2:
            try:
                extracted[f'area low {bhk}'] = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area high {bhk}'] = float(area_parts[1].replace(',', '').replace(' sq.ft.', '').strip())
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None

        # Parsing price details
        price_range = detail.get('price-range', '')
        price_parts = price_range.split('-')
        if len(price_parts) == 2:
            try:
                extracted[f'price low {bhk}'] = float(price_parts[0].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                extracted[f'price high {bhk}'] = float(price_parts[1].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                if 'L' in price_parts[0]:
                    extracted[f'price low {bhk}'] /= 100
                if 'L' in price_parts[1]:
                    extracted[f'price high {bhk}'] /= 100
            except:
                extracted[f'price low {bhk}'] = None
                extracted[f'price high {bhk}'] = None

    return extracted
# Apply the refined parsing and generate the new DataFrame structure
data_refined = []

for _, row in df_appartments.iterrows():
    features = refined_parse_modified_v2(row['PriceDetails'])
    
    # Construct a new row for the transformed dataframe
    new_row = {'PropertyName': row['PropertyName']}
    
    # Populate the new row with extracted features
    for config in ['1 BHK', '2 BHK', '3 BHK', '4 BHK', '5 BHK', '6 BHK', '1 RK', 'Land']:
        new_row[f'building type_{config}'] = features.get(f'building type_{config}')
        new_row[f'area low {config}'] = features.get(f'area low {config}')
        new_row[f'area high {config}'] = features.get(f'area high {config}')
        new_row[f'price low {config}'] = features.get(f'price low {config}')
        new_row[f'price high {config}'] = features.get(f'price high {config}')
    
    data_refined.append(new_row)

df_final_refined_v2 = pd.DataFrame(data_refined).set_index('PropertyName')

In [350]:
df_final_refined_v2

,building type_1 BHK,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,building type_2 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,building type_3 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,building type_4 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,building type_5 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,building type_6 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,building type_1 RK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,building type_Land,area low Land,area high Land,price low Land,price high Land
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,None,NaN,NaN,NaN,NaN,Apartment,1370.0,1370.0,2.0000,2.40,Apartment,1850.0,2050.0,2.25,3.59,Apartment,2600.0,2600.0,3.24,4.56,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
M3M Crown,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Apartment,1605.0,2170.0,2.20,3.03,Apartment,2248.0,2670.0,3.08,3.73,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Adani Brahma Samsara Vilasa,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Independent Floor,1800.0,3150.0,2.43,15.75,Independent Floor,2750.0,4500.0,3.36,22.50,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,,500.0,4329.0,2.05,41.13
Sobha City,None,NaN,NaN,NaN,NaN,Apartment,1381.0,1692.0,1.5500,3.21,Apartment,1711.0,2343.0,1.76,4.79,Apartment,2423.0,2963.0,2.50,6.06,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Signature Global City 93,None,NaN,NaN,NaN,NaN,Independent Floor,981.0,1118.0,0.9301,1.06,Independent Floor,1235.0,1530.0,1.12,1.45,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,None,NaN,NaN,NaN,NaN,Apartment,964.0,964.0,NaN,NaN,Apartment,1127.0,1127.0,NaN,NaN,Apartment,1562.0,1562.0,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Pyramid Urban Homes 2,Apartment,335.0,398.0,23.45,0.2786,Apartment,500.0,625.0,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Satya The Hermitage,None,NaN,NaN,NaN,NaN,Apartment,1450.0,1450.0,NaN,NaN,Apartment,1991.0,1991.0,NaN,NaN,Apartment,2639.0,4711.0,1.20,2.14,Apartment,4731.0,4731.0,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN


In [351]:
df_final_refined_v2.columns

Index(['building type_1 BHK', 'area low 1 BHK', 'area high 1 BHK',
       'price low 1 BHK', 'price high 1 BHK', 'building type_2 BHK',
       'area low 2 BHK', 'area high 2 BHK', 'price low 2 BHK',
       'price high 2 BHK', 'building type_3 BHK', 'area low 3 BHK',
       'area high 3 BHK', 'price low 3 BHK', 'price high 3 BHK',
       'building type_4 BHK', 'area low 4 BHK', 'area high 4 BHK',
       'price low 4 BHK', 'price high 4 BHK', 'building type_5 BHK',
       'area low 5 BHK', 'area high 5 BHK', 'price low 5 BHK',
       'price high 5 BHK', 'building type_6 BHK', 'area low 6 BHK',
       'area high 6 BHK', 'price low 6 BHK', 'price high 6 BHK',
       'building type_1 RK', 'area low 1 RK', 'area high 1 RK',
       'price low 1 RK', 'price high 1 RK', 'building type_Land',
       'area low Land', 'area high Land', 'price low Land', 'price high Land'],
      dtype='object')

In [352]:
df_final_refined_v2.shape

(246, 40)

In [353]:
df['PriceDetails'][0]

"{'2 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,370 sq.ft.', 'price-range': '₹ 2 - 2.4 Cr'}, '3 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,850 - 2,050 sq.ft.', 'price-range': '₹ 2.25 - 3.59 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '2,600 sq.ft.', 'price-range': '₹ 3.24 - 4.56 Cr'}}"

In [354]:
df_final_refined_v2.iloc[0]

building type_1 BHK         None
area low 1 BHK               NaN
area high 1 BHK              NaN
price low 1 BHK              NaN
price high 1 BHK             NaN
building type_2 BHK    Apartment
area low 2 BHK            1370.0
area high 2 BHK           1370.0
price low 2 BHK              2.0
price high 2 BHK             2.4
building type_3 BHK    Apartment
area low 3 BHK            1850.0
area high 3 BHK           2050.0
price low 3 BHK             2.25
price high 3 BHK            3.59
building type_4 BHK    Apartment
area low 4 BHK            2600.0
area high 4 BHK           2600.0
price low 4 BHK             3.24
price high 4 BHK            4.56
building type_5 BHK         None
area low 5 BHK               NaN
area high 5 BHK              NaN
price low 5 BHK              NaN
price high 5 BHK             NaN
building type_6 BHK         None
area low 6 BHK               NaN
area high 6 BHK              NaN
price low 6 BHK              NaN
price high 6 BHK             NaN
building t

In [355]:
categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()

In [356]:
categorical_columns

['building type_1 BHK',
 'building type_2 BHK',
 'building type_3 BHK',
 'building type_4 BHK',
 'building type_5 BHK',
 'building type_6 BHK',
 'building type_1 RK',
 'building type_Land']

In [357]:
ohe_df = pd.get_dummies(df_final_refined_v2, columns=categorical_columns, drop_first=True)

In [358]:
ohe_df.fillna(0,inplace=True)

In [359]:
ohe_df

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,area low Land,area high Land,price low Land,price high Land,building type_1 BHK_Service Apartment,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,0.0,0.0,0.00,0.0000,1370.0,1370.0,2.0000,2.40,1850.0,2050.0,2.25,3.59,2600.0,2600.0,3.24,4.56,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
M3M Crown,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1605.0,2170.0,2.20,3.03,2248.0,2670.0,3.08,3.73,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Adani Brahma Samsara Vilasa,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1800.0,3150.0,2.43,15.75,2750.0,4500.0,3.36,22.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,500.0,4329.0,2.05,41.13,False,False,False,True,False,False,True,False,False,False,False
Sobha City,0.0,0.0,0.00,0.0000,1381.0,1692.0,1.5500,3.21,1711.0,2343.0,1.76,4.79,2423.0,2963.0,2.50,6.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Signature Global City 93,0.0,0.0,0.00,0.0000,981.0,1118.0,0.9301,1.06,1235.0,1530.0,1.12,1.45,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,True,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,0.0,0.0,0.00,0.0000,964.0,964.0,0.0000,0.00,1127.0,1127.0,0.00,0.00,1562.0,1562.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Pyramid Urban Homes 2,335.0,398.0,23.45,0.2786,500.0,625.0,0.0000,0.00,0.0,0.0,0.00,0.00,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Satya The Hermitage,0.0,0.0,0.00,0.0000,1450.0,1450.0,0.0000,0.00,1991.0,1991.0,0.00,0.00,2639.0,4711.0,1.20,2.14,4731.0,4731.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False


In [360]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
ohe_df_normalized = pd.DataFrame(scaler.fit_transform(ohe_df), columns=ohe_df.columns, index=ohe_df.index)

In [361]:
ohe_df_normalized.head()

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,area low Land,area high Land,price low Land,price high Land,building type_1 BHK_Service Apartment,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,-0.252266,-0.169584,-0.105197,-0.082332,1.223499,1.020101,-0.173712,1.158423,0.553787,0.370864,0.807098,0.515061,0.602838,0.212073,0.383381,0.242019,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
M3M Crown,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.293086,0.472749,0.772095,0.342903,0.382746,0.243172,0.341443,0.108677,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Adani Brahma Samsara Vilasa,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.500583,1.304803,0.933110,4.253341,0.696627,1.056178,0.414835,3.124152,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,0.369534,1.828502,0.643875,7.635886,-0.111111,-0.289310,-0.063888,2.683282,-0.063888,-0.171139,3.924283,-0.236208,-0.111111,-0.216353,-0.063888
Sobha City,-0.252266,-0.169584,-0.105197,-0.082332,1.240497,1.470610,-0.198425,1.680336,0.405879,0.619632,0.464065,0.883970,0.492166,0.373342,0.189416,0.483000,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Signature Global City 93,-0.252266,-0.169584,-0.105197,-0.082332,0.622383,0.667529,-0.232468,0.295011,-0.100626,-0.070634,0.016022,-0.142827,-1.022842,-0.943016,-0.465873,-0.490563,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,3.456497,-0.063888,2.683282,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888


In [362]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix
cosine_sim2 = cosine_similarity(ohe_df_normalized)

In [363]:
cosine_sim2.shape

(246, 246)

In [364]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim2[ohe_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = ohe_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df


In [365]:
recommend_properties_with_scores('M3M Golf Hills')

,PropertyName,SimilarityScore
0,AIPL The Peaceful Homes,0.955462
1,Smartworld One DXP,0.954670
2,Unitech Escape,0.953092
3,M3M Capital,0.951156
4,BPTP Terra,0.943128
...,...,...
240,Golden Park,-0.522391
241,Satya Merano Greens,-0.523660
242,ROF Normanton Park,-0.525129
243,BPTP Green Oaks,-0.525286


In [366]:
df[['PropertyName','LocationAdvantages']]

,PropertyName,LocationAdvantages
0,Smartworld One DXP,"{'Bajghera Road': '800 Meter', 'Palam Vihar Ha..."
1,M3M Crown,"{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N..."
2,Adani Brahma Samsara Vilasa,"{'AIPL Business Club Sector 62': '2.7 Km', 'He..."
3,Sobha City,"{'The Shikshiyan School': '2.9 KM', 'WTC Plaza..."
4,Signature Global City 93,"{'Pranavananda Int. School': '450 m', 'DLF Sit..."
...,...,...
242,DLF Princeton Estate,"{'Sector 42-43 Metro Station': '1.8 Km', 'Para..."
243,Pyramid Urban Homes 2,{'Aarvy Healthcare Super Speciality': '1.8 KM'...
244,Satya The Hermitage,"{'Dwarka Expressway': '1.2 Km', 'S N Internati..."
245,BPTP Spacio,"{'Suncity School': '0.2 Km', 'Gurugram Road': ..."


In [367]:
df[['PropertyName','LocationAdvantages']]['LocationAdvantages'][0]

"{'Bajghera Road': '800 Meter', 'Palam Vihar Halt': '2.5 KM', 'DPSG Palam Vihar': '3.1 KM', 'Park Hospital': '3.1 KM', 'Gurgaon Railway Station': '4.9 KM', 'The NorthCap University': '5.4 KM', 'Dwarka Expy': '1.2 KM', 'Hyatt Place Gurgaon Udyog Vihar': '7.7 KM', 'Dwarka Sector 21, Metro Station': '7.2 KM', 'Pacific D21 Mall': '7.4 KM', 'Indira Gandhi International Airport': '14.7 KM', 'Hamoni Golf Camp': '6.2 KM', 'Fun N Food Waterpark': '8.8 KM', 'Accenture DDC5': '9 KM'}"

In [368]:
def distance_to_meters(distance_str):
    try:
        if 'Km' in distance_str or 'KM' in distance_str:
            return float(distance_str.split()[0]) * 1000
        elif 'Meter' in distance_str or 'meter' in distance_str:
            return float(distance_str.split()[0])
        else:
            return None
    except:
        return None

In [369]:
location_matrix = {}

for _, row in df.iterrows():
    distances = {}

    for location, distance in ast.literal_eval(row['LocationAdvantages']).items():
        distances[location] = distance_to_meters(distance)

    location_matrix[row['PropertyName']] = distances

location_df = pd.DataFrame.from_dict(location_matrix, orient='index')

In [370]:
location_df.shape

(246, 1070)

In [371]:
print(location_df.columns.tolist())

['Bajghera Road', 'Palam Vihar Halt', 'DPSG Palam Vihar', 'Park Hospital', 'Gurgaon Railway Station', 'The NorthCap University', 'Dwarka Expy', 'Hyatt Place Gurgaon Udyog Vihar', 'Dwarka Sector 21, Metro Station', 'Pacific D21 Mall', 'Indira Gandhi International Airport', 'Hamoni Golf Camp', 'Fun N Food Waterpark', 'Accenture DDC5', 'DPSG Palam Vihar Gurugram', 'Park Hospital, Palam Vihar', 'Palam Vihar Halt Railway Station', 'Dwarka Sector 21 Metro Station', 'Dwarka Expressway', 'Fun N Food Water Park', 'Tau DeviLal Sports Complex', 'Hyatt Place', 'Altrade Business Centre', 'AIPL Business Club Sector 62', 'Heritage Xperiential Learning School', 'CK Birla Hospital', 'Paras Trinity Mall Sector 63', 'Rapid Metro Station Sector 56', 'De Adventure Park', 'Golf Course Ext Rd', 'DoubleTree by Hilton Hotel Gurgaon', 'KIIT College of Engineering Sohna Road', 'Mehrauli-Gurgaon Road', 'Nirvana Rd', 'TERI Golf Course', 'The Shikshiyan School', 'WTC Plaza', 'Luxus Haritma Resort', 'BSF Golf Course

In [372]:
#finding near-duplicates columns
from collections import defaultdict
import re

def normalize(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())

groups = defaultdict(list)

for col in location_df.columns:
    groups[normalize(col)].append(col)

duplicates = {k: v for k, v in groups.items() if len(v) > 1}

for v in duplicates.values():
    print(v)

['Gurgaon Railway Station', 'Gurgaon railway station']
['Dwarka Sector 21, Metro Station', 'Dwarka Sector 21 Metro Station', 'Dwarka Sector 21 Metro station', 'Dwarka sector 21 metro station']
['Fun N Food Waterpark', 'Fun N Food Water Park', 'Fun N Food WaterPark']
['Dwarka Expressway', 'Dwarka expressway']
['Tau DeviLal Sports Complex', 'Tau Devi Lal Sports Complex']
['TERI Golf Course', 'Teri Golf Course']
['Rions Hospital', "Rion's Hospital"]
['NH48', 'NH 48', 'NH-48']
['AapnoGhar', 'Aapno Ghar']
['NH -8', 'NH8', 'NH-8', 'N.H-8', 'NH 8']
['IMT Manesar', 'Imt Manesar']
['Golf Course Extension Road', 'Golf course extension road']
['Euro International School, Sector- 109', 'Euro International School, Sector- 109.']
['Pataudi Road', 'Pataudi road']
['Iris Broadway Mall', 'IRIS Broadway Mall']
['Sector 55-56 Metro Station', 'Sector 55/56 Metro Station', 'Sector 55-56 Metro station', 'Sector 55-56 metro station']
['Omaxe Gurgaon Mall', 'OMAXE Gurgaon Mall']
['National Highway  48', 'Nati

In [373]:
#finding similar columns
from rapidfuzz import fuzz
import re

# ---------- Normalize ----------
def normalize(name):
    name = name.lower()
    replacements = {
        r'\bexpy\b': 'expressway',
        r'\bextn\b': 'extension',
        r'\bext\b': 'extension',
        r'\brd\b': 'road',
        r'\bintl\b': 'international',
        r'\bint\.\b': 'international',
        r'\bstn\b': 'station',
        r'&': 'and'
    }
    for old, new in replacements.items():
        name = re.sub(old, new, name)
    # remove punctuation
    name = re.sub(r'[^a-z0-9 ]', ' ', name)
    # remove extra spaces
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# ---------- Extract Numbers ----------
def extract_numbers(text):
    return re.findall(r'\d+', text)

# ---------- Detect Similar Columns ----------
columns = location_df.columns.tolist()

visited = set()
similar_groups = []
for i, col1 in enumerate(columns):
    if col1 in visited:
        continue
    group = [col1]
    norm1 = normalize(col1)
    nums1 = extract_numbers(norm1)

    for j in range(i + 1, len(columns)):
        col2 = columns[j]
        if col2 in visited:
            continue
        norm2 = normalize(col2)
        nums2 = extract_numbers(norm2)
        
        # Numbers must match
        if nums1 != nums2:
            continue
        score = fuzz.token_sort_ratio(norm1, norm2)
        if score >= 92:
            group.append(col2)
    if len(group) > 1:
        similar_groups.append(group)
        visited.update(group)

# ---------- Print ----------
for g in similar_groups:
    print(g)

['Gurgaon Railway Station', 'Gurgaon Old Railway Station', 'Gurgaon railway station']
['Dwarka Expy', 'Dwarka Expressway', 'Dwaraka Expressway', 'Dwarka expressway']
['Dwarka Sector 21, Metro Station', 'Dwarka Sector 21 Metro Station', 'Dwarka Sector 21 Metro station', 'Dwarka sector 21 metro station']
['Indira Gandhi International Airport', 'Indira Gandhi Intl Airport']
['Fun N Food Waterpark', 'Fun N Food WaterPark']
['Tau DeviLal Sports Complex', 'Tau Devi Lal Sports Complex']
['Golf Course Ext Rd', 'Golf Course Extension Road', 'Golf Course Ext Road', 'Golf Course Extension Rd', 'Golf Corse Ext. Rd.', 'Golf Course Extn Road', 'Golf course extension road']
['Mehrauli-Gurgaon Road', 'Mehrauli-Gurgaon Rd']
['TERI Golf Course', 'Teri Golf Course']
['Rions Hospital', "Rion's Hospital"]
['IGI Airport', 'IGIA Airport']
['AapnoGhar', 'Aapno Ghar']
['NH 48', 'NH-48']
['NH -8', 'NH-8', 'NH 8']
['IMT Manesar', 'Imt Manesar']
['Euro International School, Sector- 109', 'Euro International Schoo

In [374]:
# Merge duplicate columns

for group in duplicates.values():

    # Skip if some columns were already removed
    group = [c for c in group if c in location_df.columns]

    if len(group) < 2:
        continue

    # Choose canonical name
    canonical = max(group, key=len)

    # Merge (minimum distance)
    location_df[canonical] = location_df[group].min(axis=1)

    # Drop remaining columns
    location_df.drop(columns=[c for c in group if c != canonical],
                     inplace=True)

print("Duplicate columns merged.")

Duplicate columns merged.


In [375]:
# Merge similar columns

for group in similar_groups:

    # Some columns may already have been dropped
    group = [c for c in group if c in location_df.columns]

    if len(group) < 2:
        continue

    # Choose canonical name
    canonical = max(group, key=len)

    # Merge (minimum distance)
    location_df[canonical] = location_df[group].min(axis=1)

    # Drop remaining columns
    location_df.drop(columns=[c for c in group if c != canonical],
                     inplace=True)

print("Similar columns merged.")

Similar columns merged.


In [376]:
location_df.shape

(246, 945)

In [377]:
location_df

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,The NorthCap University,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Fun N Food Water Park,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Gurgaon,Dwarka Sector 21,Nehru Stadium,Vasant Kunj,Pranavananda Int. School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,Garhi Harsaru Junction,Manesar Golf Course,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,NH 48,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109",Jai Sai Ram Hospital,...,SportsCube Center(Sports Complex),Pragyanam School,Old Bengali Market,St. Angel's Global,Ninex Mall,Oriental Bank of Commerce,Shaheed Bhagat Singh Chowk,ISKCON,Meditree Market,Leopard hills,DLF5 Summit Plaza,Kriti Hospital,Anand Multispeciality Hospital,JMD Regent Mall,Eye Doctors at Krishna Netralaya,Tagore Public School,Syndicate Bank,Star Nursery,HP Petrol Pump,Green garden narsari,Lotus Valley School,Fresco Market,Insfire Sports,Emerald Plaza,Shiva Temple Tigra,McDonald's India,M3M IFC,M3M Cosmopolitan Mall,The Sylvan Trails School,Leisure Valley Park,Hyatt Regency Hotel,Knowledge Tree World School,Presidium School Gurgoan,Manav Rachna School,Windsor International School,Dreamz Cafe,Biryani Shah,KLAY Play School,Spaze Business Park,BM College of Technology,Kheri Railway station,Unitech Business Zone,Delhi Public School Gurugram Sector 67A,Samrat Mihir Bhoj Road,AIIMS,The Hive,JMS Marine Square,Stymerra Chowk,Sector 102 Dhankot,Shri Hanuman Ji Mandir,MCC Cricket Ground Dhankot,The Shri Ram School Aravali,Taj City Centre Gurugram,Minda Industries Corporate Office,"Rampura Flyover, Naurangpur Rd",Manesar toll plaza - Kherki Daula,"Imt Manesar, Gurugram",Holiday Inn,Sector 84 Road,Skyview Corporate Park
Smartworld One DXP,800.0,2500.0,3100.0,3100.0,5400.0,7700.0,7200.0,7400.0,14700.0,6200.0,9000.0,NaN,NaN,NaN,8800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
La Vida by Tata Housing,550.0,NaN,NaN,NaN,6700.0,NaN,NaN,7500.0,15600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7400.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Oxirich Chintamanis,5300.0,NaN,NaN,NaN,8800.0,NaN,NaN,NaN,20800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [378]:
location_df.fillna(54000,inplace=True) #cannot replace by 0 as they are distances and if NaN replaced by very large no 

In [379]:
from sklearn.preprocessing import StandardScaler
# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
location_df_normalized = pd.DataFrame(scaler.fit_transform(location_df), columns=location_df.columns, index=location_df.index)

In [380]:
location_df_normalized

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,The NorthCap University,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Fun N Food Water Park,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Gurgaon,Dwarka Sector 21,Nehru Stadium,Vasant Kunj,Pranavananda Int. School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,Garhi Harsaru Junction,Manesar Golf Course,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,NH 48,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109",Jai Sai Ram Hospital,...,SportsCube Center(Sports Complex),Pragyanam School,Old Bengali Market,St. Angel's Global,Ninex Mall,Oriental Bank of Commerce,Shaheed Bhagat Singh Chowk,ISKCON,Meditree Market,Leopard hills,DLF5 Summit Plaza,Kriti Hospital,Anand Multispeciality Hospital,JMD Regent Mall,Eye Doctors at Krishna Netralaya,Tagore Public School,Syndicate Bank,Star Nursery,HP Petrol Pump,Green garden narsari,Lotus Valley School,Fresco Market,Insfire Sports,Emerald Plaza,Shiva Temple Tigra,McDonald's India,M3M IFC,M3M Cosmopolitan Mall,The Sylvan Trails School,Leisure Valley Park,Hyatt Regency Hotel,Knowledge Tree World School,Presidium School Gurgoan,Manav Rachna School,Windsor International School,Dreamz Cafe,Biryani Shah,KLAY Play School,Spaze Business Park,BM College of Technology,Kheri Railway station,Unitech Business Zone,Delhi Public School Gurugram Sector 67A,Samrat Mihir Bhoj Road,AIIMS,The Hive,JMS Marine Square,Stymerra Chowk,Sector 102 Dhankot,Shri Hanuman Ji Mandir,MCC Cricket Ground Dhankot,The Shri Ram School Aravali,Taj City Centre Gurugram,Minda Industries Corporate Office,"Rampura Flyover, Naurangpur Rd",Manesar toll plaza - Kherki Daula,"Imt Manesar, Gurugram",Holiday Inn,Sector 84 Road,Skyview Corporate Park
Smartworld One DXP,-7.960979,-15.652476,-15.652476,-3.149592,-3.147217,-10.231739,-5.396042,-6.023233,-1.425834,-5.664722,-15.652476,0.063888,0.090533,0.063888,-7.630549,0.063888,0.090382,0.063888,0.158102,0.144014,0.063888,0.063888,0.254151,0.090283,0.063888,0.11086,0.063888,0.143552,0.144016,0.111073,0.090536,0.090536,0.090494,0.090501,0.143948,0.063888,0.0,0.063888,0.253961,0.063888,0.063888,0.28905,0.353525,0.143984,0.063888,0.157949,0.171008,0.063888,0.216148,0.090501,0.063888,0.090535,0.143929,0.111108,0.128333,0.304364,0.063888,0.335088,0.063888,0.090522,...,0.063888,0.063888,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.0,0.0,0.0,0.0,0.0,0.063888,0.063888,0.0,0.0,0.063888,0.063888,0.063888,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.0,0.063888,0.063888
La Vida by Tata Housing,-7.998993,0.063888,0.063888,0.328277,-3.054053,0.090308,0.194781,-6.009941,-1.365333,0.182906,0.063888,0.063888,0.090533,0.063888,0.128253,0.063888,0.090382,0.063888,0.158102,0.144014,0.063888,0.063888,0.254151,0.090283,0.063888,0.11086,0.063888,0.143552,0.144016,0.111073,0.090536,0.090536,0.090494,-11.347959,0.143948,0.063888,0.0,0.063888,0.253961,0.063888,0.063888,0.28905,0.353525,0.143984,0.063888,0.157949,0.171008,0.063888,0.216148,0.090501,0.063888,0.090535,0.143929,0.

In [381]:
cosine_sim3 = cosine_similarity(location_df_normalized)

In [382]:
cosine_sim3.shape

(246, 246)

In [383]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim3[location_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df

In [384]:
recommend_properties_with_scores('BPTP Terra')

,PropertyName,SimilarityScore
0,AIPL The Peaceful Homes,0.129436
1,M3M Merlin,0.103459
2,Tulip Yellow,0.101208
3,Bestech Park View Sanskruti,0.092462
4,International City by SOBHA Phase 2,0.090292
...,...,...
240,Ansal API Esencia,-0.054233
241,The Close South,-0.054233
242,DLF New Town Heights 2,-0.054233
243,Antriksh Heights,-0.054233


In [385]:
3*cosine_sim3 + 5*cosine_sim2 + 6*cosine_sim1

array([[14.        ,  1.19133255,  0.21727255, ...,  1.2212103 ,
         0.75505959,  1.79359813],
       [ 1.19133255, 14.        ,  1.06887686, ..., -0.73456689,
        -0.78415277, -0.74611071],
       [ 0.21727255,  1.06887686, 14.        , ..., -0.7856021 ,
        -1.95023916, -1.61287042],
       ...,
       [ 1.2212103 , -0.73456689, -0.7856021 , ..., 14.        ,
         1.30266004,  1.29813121],
       [ 0.75505959, -0.78415277, -1.95023916, ...,  1.30266004,
        14.        ,  7.81050775],
       [ 1.79359813, -0.74611071, -1.61287042, ...,  1.29813121,
         7.81050775, 14.        ]], shape=(246, 246))

In [386]:
def recommend_properties_with_scores(property_name, top_n=247):

    cosine_sim_matrix = 30 * cosine_sim1 + 20 * cosine_sim2 + 8 * cosine_sim3
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df


In [387]:
recommend_properties_with_scores('BPTP Terra')

,PropertyName,SimilarityScore
0,Lion Infra Green Valley,22.179061
1,Pegasus Atulyam 83,21.481893
2,Ireo The Corridors,19.999230
3,Conscient Habitat,19.023959
4,Bestech Park View Spa Next,18.681379
...,...,...
240,Emaar Palm Gardens,-9.048380
241,M3M Skywalk,-9.233710
242,Vatika Aspiration,-9.293958
243,Orchid Petals,-9.533701


In [395]:
import pickle
pickle.dump(location_df,open('location_distance.pkl','wb'))

In [396]:
pickle.dump(cosine_sim1, open('cosine_sim1.pkl', 'wb'))
pickle.dump(cosine_sim2, open('cosine_sim2.pkl', 'wb'))
pickle.dump(cosine_sim3, open('cosine_sim3.pkl', 'wb'))

In [400]:
apartment_link=df[['PropertyName','Link']]

In [401]:
pickle.dump(apartment_link,open('apartment_link.pkl','wb'))